# Anomaly Detection & Time Series — Assignment Solutions
### PW Skills — Data Science with GenAI | Python ML Module
**Assignment Code:** DA-AG-018 





In [ ]:
# If statsmodels is not already installed in your environment, uncomment and run:
# !pip install statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.datasets import make_blobs

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Part A — Theoretical Questions

### Q1. What is Anomaly Detection? Explain its types (point, contextual, and collective anomalies) with examples.

**Anomaly Detection** is the process of identifying data points, events, or observations that deviate significantly from the expected/normal pattern in a dataset. These unusual observations are called **anomalies** or **outliers**, and detecting them is important in domains like fraud detection, network security, manufacturing defect detection, and healthcare monitoring.

There are three main types of anomalies:

1. **Point Anomalies**: A single data instance that is anomalous with respect to the rest of the data.
   - *Example*: A credit card transaction of ₹5,00,000 when a customer's typical spend is ₹2,000–₹5,000.

2. **Contextual Anomalies**: A data instance that is anomalous only in a specific **context** (e.g., time, location), but would be normal in another context.
   - *Example*: A temperature of 35°C is normal in summer but would be a contextual anomaly if recorded in the middle of winter.

3. **Collective Anomalies**: A **group/collection** of related data instances that is anomalous with respect to the entire dataset, even though the individual points within the group may not be anomalous on their own.
   - *Example*: A sudden, sustained sequence of unusually low heart-rate readings over 10 minutes in ECG data — each single reading might be within a plausible range, but the sustained pattern together indicates an anomaly (e.g., a medical event).

### Q2. Compare Isolation Forest, DBSCAN, and Local Outlier Factor in terms of their approach and suitable use cases.

| Aspect | Isolation Forest | DBSCAN | Local Outlier Factor (LOF) |
|---|---|---|---|
| **Approach** | Builds random binary trees that recursively split the data; anomalies require **fewer splits to isolate** (shorter average path length) since they are "few and different." | A **density-based clustering** algorithm; points that don't belong to any dense region (not reachable from a core point) are labeled as **noise/outliers**. | Compares the **local density** of a point to the local densities of its k-nearest neighbors; points with **substantially lower density** than their neighbors get a high LOF score (outliers). |
| **Anomaly definition** | Global — based on how easily a point is isolated from the rest of the data. | Implicit — any point not part of a dense cluster is an outlier. | Local — a point can be an outlier only relative to its neighborhood, even if it's not a global outlier. |
| **Handles varying density** | Yes, reasonably well; not density-based so less affected. | No — struggles when clusters have different densities (fixed `eps`/`min_samples`). | Yes — this is LOF's key strength; explicitly designed for varying-density regions. |
| **Scalability** | Very good — O(n log n), works well on large/high-dimensional data. | Moderate — can be slow on very large datasets. | Slower — requires nearest-neighbor computation for every point, less scalable to large datasets. |
| **Best use cases** | Large, high-dimensional datasets; general-purpose outlier detection (e.g., fraud detection, network intrusion). | Data with well-separated dense clusters and roughly uniform density (e.g., spatial/geographic clustering with outlier detection as a byproduct). | Datasets with **clusters of varying density** where local context matters (e.g., detecting local anomalies in sensor networks, region-specific fraud patterns). |


### Q3. What are the key components of a Time Series? Explain each with one example.

A time series can typically be decomposed into four key components:

1. **Trend**: The long-term increase or decrease in the data over time.
   - *Example*: A steady rise in monthly airline passenger numbers over the years due to growing air travel demand.

2. **Seasonality**: Regular, repeating patterns or fluctuations at fixed intervals (daily, weekly, monthly, yearly) due to seasonal factors.
   - *Example*: Retail sales spiking every year in November–December due to holiday shopping.

3. **Cyclic variations**: Fluctuations that occur over **longer, irregular periods** (not fixed like seasonality), often related to economic or business cycles.
   - *Example*: Real-estate prices rising and falling over multi-year economic boom-and-bust cycles.

4. **Residual / Irregular component (Noise)**: The random, unpredictable variation left over after removing trend, seasonality, and cyclic effects.
   - *Example*: A sudden one-off dip in sales due to an unexpected local event, which is not part of any repeating or systematic pattern.

### Q4. Define Stationary in time series. How can you test and transform a non-stationary series into a stationary one?

A time series is **stationary** if its statistical properties — **mean, variance, and autocovariance** — remain **constant over time** (do not depend on the time at which they are observed). Stationarity is an important assumption for many classical forecasting models (like ARIMA), because they rely on consistent statistical structure to make reliable predictions.

**Testing for stationarity:**
- **Visual inspection**: Plot the series and rolling mean/variance — a stationary series looks flat with constant spread; a non-stationary one shows visible trend/seasonality or changing variance.
- **Augmented Dickey-Fuller (ADF) test**: A statistical hypothesis test where the null hypothesis is that the series is non-stationary (has a unit root). A small p-value (typically < 0.05) lets us reject the null and conclude the series is stationary.
- **KPSS test**: The opposite null hypothesis (series is stationary) — often used alongside ADF to cross-check.

**Transforming a non-stationary series into a stationary one:**
- **Differencing**: Subtracting the previous observation from the current one (`y_t - y_{t-1}`) to remove trend; can apply **seasonal differencing** (`y_t - y_{t-m}`) to remove seasonality.
- **Log transformation**: Applying `log(y_t)` to stabilize/reduce increasing variance over time.
- **Detrending**: Fitting and subtracting a trend line (e.g., using regression) from the series.
- **Combination**: Often log transform + differencing (and seasonal differencing) together are used, as in the "airline model" ARIMA(0,1,1)(0,1,1).

### Q5. Differentiate between AR, MA, ARIMA, SARIMA, and SARIMAX models in terms of structure and application.

| Model | Structure | Application |
|---|---|---|
| **AR (Autoregressive)** | Predicts the current value as a **linear combination of its own past values**: `y_t = c + φ₁y_{t-1} + φ₂y_{t-2} + ... + φₚy_{t-p} + εₜ`. Order `p`. | Used when the series shows momentum/dependency on its own recent past (e.g., stock returns with autocorrelation). |
| **MA (Moving Average)** | Predicts the current value as a linear combination of **past forecast errors (residuals)**: `y_t = c + εₜ + θ₁ε_{t-1} + ... + θ_qε_{t-q}`. Order `q`. | Used when shocks/random events have a lingering short-term effect on future values. |
| **ARIMA (AutoRegressive Integrated Moving Average)** | Combines AR(p) + differencing "I" of order `d` (to make the series stationary) + MA(q): **ARIMA(p, d, q)**. | General-purpose forecasting for **non-seasonal, non-stationary** time series (e.g., trending sales data without a strong seasonal pattern). |
| **SARIMA (Seasonal ARIMA)** | Extends ARIMA with **seasonal terms**: **SARIMA(p, d, q)(P, D, Q, m)**, where `P, D, Q` are the seasonal AR/differencing/MA orders and `m` is the season length (e.g., 12 for monthly data with yearly seasonality). | Used for series with **both trend and clear seasonality** (e.g., monthly airline passengers, retail sales with yearly seasonal cycles). |
| **SARIMAX (SARIMA with eXogenous variables)** | Same as SARIMA but also incorporates **external/exogenous predictor variables** that influence the target series: `y_t` depends on its own history **plus** external regressors `X_t` (e.g., weather, holidays, promotions). | Used when forecasting accuracy can be improved with additional known/observable factors (e.g., forecasting energy demand using temperature as an exogenous variable). |

**In short**: AR and MA are simple building blocks → ARIMA combines them with differencing for non-seasonal data → SARIMA adds seasonal structure → SARIMAX further adds external explanatory variables.

## Part B — Practical Questions

### Q6. Load a time series dataset (e.g., AirPassengers), plot the original series, and decompose it into trend, seasonality, and residual components.

In [ ]:
# Classic AirPassengers dataset (monthly totals of international airline passengers, 1949-1960, in thousands)
# Hard-coded here so the notebook runs fully offline without needing to download AirPassengers.csv
airpassengers_values = [
    112,118,132,129,121,135,148,148,136,119,104,118,   # 1949
    115,126,141,135,125,149,170,170,158,133,114,140,   # 1950
    145,150,178,163,172,178,199,199,184,162,146,166,   # 1951
    171,180,193,181,183,218,230,242,209,191,172,194,   # 1952
    196,196,236,235,229,243,264,272,237,211,180,201,   # 1953
    204,188,235,227,234,264,302,293,259,229,203,229,   # 1954
    242,233,267,269,270,315,364,347,312,274,237,278,   # 1955
    284,277,317,313,318,374,413,405,355,306,271,306,   # 1956
    315,301,356,348,355,422,465,467,404,347,305,336,   # 1957
    340,318,362,348,363,435,491,505,404,359,310,337,   # 1958
    360,342,406,396,420,472,548,559,463,407,362,405,   # 1959
    417,391,419,461,472,535,622,606,508,461,390,432,   # 1960
]

date_index = pd.date_range(start='1949-01-01', periods=len(airpassengers_values), freq='MS')
air_passengers = pd.Series(airpassengers_values, index=date_index, name='Passengers')

# Plot the original series
plt.figure(figsize=(11,4))
plt.plot(air_passengers, color='steelblue')
plt.title("AirPassengers - Monthly International Airline Passengers (1949-1960)")
plt.xlabel("Year")
plt.ylabel("Passengers (thousands)")
plt.grid(True, alpha=0.3)
plt.show()

# Decompose into trend, seasonality, and residual (multiplicative, since variance grows with level)
decomposition = seasonal_decompose(air_passengers, model='multiplicative', period=12)

fig = decomposition.plot()
fig.set_size_inches(11, 8)
plt.tight_layout()
plt.show()


### Q7. Apply Isolation Forest on a numerical dataset (e.g., NYC Taxi Fare) to detect anomalies. Visualize the anomalies on a 2D scatter plot.

In [ ]:
# Synthetic NYC-taxi-fare-like dataset: trip_distance (miles) vs fare_amount ($)
# (Mirrors the structure of the real Kaggle NYC Taxi Fare dataset; replace with pd.read_csv('train.csv') for the real data)
n_rides = 1000
trip_distance = np.random.uniform(0.5, 20, n_rides)
base_fare = 2.5 + 2.5 * trip_distance                      # normal fare relationship
fare_amount = base_fare + np.random.normal(0, 2, n_rides)  # add realistic noise

# Inject some anomalies: unusually high fares for short trips, and near-zero fares for long trips
n_anomalies = 25
anomaly_idx = np.random.choice(n_rides, n_anomalies, replace=False)
for i in anomaly_idx[:15]:
    trip_distance[i] = np.random.uniform(0.5, 3)
    fare_amount[i] = np.random.uniform(80, 200)   # suspiciously high fare for a short trip
for i in anomaly_idx[15:]:
    trip_distance[i] = np.random.uniform(15, 20)
    fare_amount[i] = np.random.uniform(1, 5)      # suspiciously low fare for a long trip

taxi_df = pd.DataFrame({'trip_distance': trip_distance, 'fare_amount': fare_amount})

iso_forest = IsolationForest(contamination=0.03, random_state=RANDOM_STATE)
taxi_df['anomaly'] = iso_forest.fit_predict(taxi_df[['trip_distance', 'fare_amount']])
# anomaly == -1 -> outlier, anomaly == 1 -> normal

plt.figure(figsize=(7,6))
normal_pts = taxi_df[taxi_df['anomaly'] == 1]
outlier_pts = taxi_df[taxi_df['anomaly'] == -1]

plt.scatter(normal_pts['trip_distance'], normal_pts['fare_amount'], c='steelblue', s=25, label='Normal')
plt.scatter(outlier_pts['trip_distance'], outlier_pts['fare_amount'], c='red', marker='x', s=60, label='Anomaly')
plt.xlabel("Trip Distance (miles)")
plt.ylabel("Fare Amount ($)")
plt.title("Isolation Forest - Anomaly Detection on NYC Taxi Fare Data")
plt.legend()
plt.show()

print(f"Number of anomalies detected: {(taxi_df['anomaly'] == -1).sum()} out of {len(taxi_df)} rides")


### Q8. Train a SARIMA model on the monthly airline passengers dataset. Forecast the next 12 months and visualize the results.

In [ ]:
# Using the classic Box-Jenkins "airline model": SARIMA(0,1,1)(0,1,1,12)
sarima_model = SARIMAX(air_passengers,
                        order=(0, 1, 1),
                        seasonal_order=(0, 1, 1, 12),
                        enforce_stationarity=False,
                        enforce_invertibility=False)
sarima_fit = sarima_model.fit(disp=False)

print(sarima_fit.summary())

# Forecast the next 12 months
forecast_steps = 12
forecast_result = sarima_fit.get_forecast(steps=forecast_steps)
forecast_mean = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int()

plt.figure(figsize=(11,5))
plt.plot(air_passengers, label='Observed', color='steelblue')
plt.plot(forecast_mean, label='Forecast (next 12 months)', color='darkorange')
plt.fill_between(forecast_ci.index, forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1],
                  color='orange', alpha=0.2, label='95% Confidence Interval')
plt.title("SARIMA Forecast - AirPassengers (Next 12 Months)")
plt.xlabel("Year")
plt.ylabel("Passengers (thousands)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nForecasted values for the next 12 months:")
print(forecast_mean)


### Q9. Apply Local Outlier Factor (LOF) on any numerical dataset to detect anomalies and visualize them using matplotlib.

In [ ]:
# Generate a 2D dataset with clusters of varying density, plus a handful of scattered outliers
X_lof, _ = make_blobs(n_samples=300, centers=3, cluster_std=[0.5, 1.0, 1.5], random_state=RANDOM_STATE)
rng = np.random.RandomState(RANDOM_STATE)
outliers_lof = rng.uniform(low=X_lof.min() - 4, high=X_lof.max() + 4, size=(20, 2))
X_lof_full = np.vstack([X_lof, outliers_lof])

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.06)
lof_labels = lof.fit_predict(X_lof_full)   # -1 = outlier, 1 = inlier

plt.figure(figsize=(7,6))
plt.scatter(X_lof_full[lof_labels == 1, 0], X_lof_full[lof_labels == 1, 1],
            c='steelblue', s=25, label='Normal')
plt.scatter(X_lof_full[lof_labels == -1, 0], X_lof_full[lof_labels == -1, 1],
            c='red', marker='x', s=60, label='Anomaly (LOF)')
plt.title("Local Outlier Factor (LOF) - Anomaly Detection")
plt.legend()
plt.show()

print(f"Number of anomalies detected by LOF: {(lof_labels == -1).sum()} out of {len(lof_labels)} points")


### Q10. Power Grid Monitoring — Real-Time Data Science Workflow

**Scenario**: Forecast energy demand and detect abnormal spikes/drops in real-time consumption data collected every 15 minutes, with features like timestamp, region, weather conditions, and energy usage.

**Answer:**

**1. How would I detect anomalies in this streaming data (Isolation Forest / LOF / DBSCAN)?**
I would use **Isolation Forest** as the primary detector for this use case:
- It scales well to high-frequency streaming data (15-min intervals → ~96 points/day/region, easily thousands of points across regions).
- It naturally handles **multivariate** anomaly detection — combining energy usage with weather features (temperature, humidity) and time-based features (hour of day, day of week) to catch **contextual anomalies** (e.g., high usage at 3 AM is anomalous even if the same value would be normal at 6 PM).
- It can be retrained/refit periodically (e.g., daily) on a rolling window of recent data to adapt to evolving "normal" behavior (seasonal changes, new equipment, etc.).
- **LOF** could be used as a secondary/complementary check for region-specific anomalies, since different regions may have very different baseline densities/consumption patterns (LOF handles varying density well). **DBSCAN** is less suited here since it doesn't easily support the continuous, streaming, high-dimensional nature of this problem and needs stable `eps`/`min_samples` tuning.

**2. Which time series model would I use for short-term forecasting (ARIMA / SARIMA / SARIMAX)?**
I would use **SARIMAX**:
- Energy demand has strong **daily seasonality** (period = 96 intervals at 15-min resolution, or 24 at hourly resolution) and often **weekly seasonality** (weekday vs weekend usage patterns).
- SARIMAX additionally lets me incorporate **exogenous variables** like temperature/weather conditions and calendar features (holiday flags), which are known to strongly influence energy demand — plain ARIMA/SARIMA can't use this extra information.
- For very short-term (next few hours) operational forecasting, SARIMAX combined with a rolling/expanding-window refit strategy gives accurate, explainable forecasts; for longer horizons or highly non-linear demand patterns, this could be supplemented with gradient boosting or LSTM-based models.

**3. How would I validate and monitor performance over time?**
- Use **rolling-window (walk-forward) backtesting**: train on a window of historical data, forecast the next period, compare to actuals, then slide the window forward — this mimics real deployment.
- Track forecasting error metrics over time (**MAE, RMSE, MAPE**) on a dashboard, and set up **alerts** if error exceeds a threshold (indicating model drift).
- For anomaly detection, track the **anomaly rate** over time (should stay roughly stable); a sudden jump could indicate either a genuine grid issue or that the model needs retraining.
- Periodically **retrain** both the anomaly detector and the SARIMAX model on recent data (e.g., weekly) to adapt to seasonal/behavioral drift, and use a held-out validation set each time to confirm the new model isn't worse than the old one before deploying it.

**4. How would this solution help business decisions or operations?**
- **Proactive maintenance**: Detected anomalies (sudden spikes/drops) can trigger alerts to field engineers to investigate equipment faults or outages *before* they cascade into larger failures.
- **Demand-supply balancing**: Accurate short-term forecasts let the grid operator schedule power generation/purchases efficiently, reducing costs from over- or under-provisioning.
- **Fraud/theft detection**: Unusual consumption drops (possible meter tampering) or spikes (illegal tapping) can be flagged for investigation.
- **Customer experience**: Faster detection of outages/abnormal patterns per region enables quicker response and communication to affected customers, improving reliability and trust.

Below is a small illustrative pipeline putting this together on synthetic 15-minute interval energy data.


In [ ]:
# --- Illustrative pipeline on synthetic 15-minute power grid data ---

# 1. Generate synthetic 15-min interval energy demand data for 14 days (14*96 = 1344 points)
n_days = 14
points_per_day = 96  # 24h * 4 (15-min intervals)
n_points = n_days * points_per_day

timestamps = pd.date_range(start='2026-06-01', periods=n_points, freq='15min')
time_of_day = np.tile(np.arange(points_per_day), n_days)

# Daily seasonal pattern (higher demand in morning/evening, lower at night) + weekly effect + slight upward trend + noise
daily_pattern = 50 + 30 * np.sin((time_of_day / points_per_day) * 2 * np.pi - np.pi/2) ** 2
trend = np.linspace(0, 10, n_points)
weekday_effect = np.where(timestamps.dayofweek < 5, 5, -5)  # weekdays slightly higher demand
noise = np.random.normal(0, 3, n_points)

energy_usage = daily_pattern + trend + weekday_effect + noise
energy_usage = np.clip(energy_usage, 10, None)

# Inject a handful of anomalies (sudden spikes/drops)
anomaly_positions = np.random.choice(n_points, 10, replace=False)
for pos in anomaly_positions[:5]:
    energy_usage[pos] *= np.random.uniform(1.8, 2.5)   # spike
for pos in anomaly_positions[5:]:
    energy_usage[pos] *= np.random.uniform(0.1, 0.3)   # drop

grid_df = pd.DataFrame({'timestamp': timestamps, 'energy_usage': energy_usage})
grid_df.set_index('timestamp', inplace=True)

# 2. Anomaly detection with Isolation Forest (using usage + hour-of-day as context)
grid_df['hour'] = grid_df.index.hour + grid_df.index.minute / 60
iso_grid = IsolationForest(contamination=0.01, random_state=RANDOM_STATE)
grid_df['anomaly'] = iso_grid.fit_predict(grid_df[['energy_usage', 'hour']])

plt.figure(figsize=(12,4))
plt.plot(grid_df.index, grid_df['energy_usage'], color='steelblue', label='Energy Usage')
plt.scatter(grid_df.index[grid_df['anomaly'] == -1], grid_df.loc[grid_df['anomaly'] == -1, 'energy_usage'],
            color='red', marker='x', s=60, label='Detected Anomaly')
plt.title("Power Grid: 15-min Energy Usage with Detected Anomalies (Isolation Forest)")
plt.xlabel("Time")
plt.ylabel("Energy Usage (kWh)")
plt.legend()
plt.show()

# 3. Short-term forecasting with SARIMAX (aggregate to hourly for a faster, more interpretable fit)
hourly_usage = grid_df['energy_usage'].resample('h').mean()

sarimax_grid = SARIMAX(hourly_usage, order=(1, 1, 1), seasonal_order=(1, 1, 1, 24),
                        enforce_stationarity=False, enforce_invertibility=False)
sarimax_grid_fit = sarimax_grid.fit(disp=False)

forecast_hours = 24  # forecast next 24 hours
grid_forecast = sarimax_grid_fit.get_forecast(steps=forecast_hours)
grid_forecast_mean = grid_forecast.predicted_mean
grid_forecast_ci = grid_forecast.conf_int()

plt.figure(figsize=(12,4))
plt.plot(hourly_usage, label='Observed (hourly avg)', color='steelblue')
plt.plot(grid_forecast_mean, label='Forecast (next 24h)', color='darkorange')
plt.fill_between(grid_forecast_ci.index, grid_forecast_ci.iloc[:, 0], grid_forecast_ci.iloc[:, 1],
                  color='orange', alpha=0.2, label='95% CI')
plt.title("SARIMAX Short-Term Forecast - Power Grid Energy Demand (Next 24 Hours)")
plt.xlabel("Time")
plt.ylabel("Energy Usage (kWh)")
plt.legend()
plt.show()

print(f"Detected {int((grid_df['anomaly'] == -1).sum())} anomalies out of {len(grid_df)} readings.")


## Summary

This notebook covered:
- **5 theoretical questions** explaining anomaly detection (point/contextual/collective anomalies), a comparison of Isolation Forest, DBSCAN and LOF, the key components of a time series, stationarity testing/transformation, and the differences between AR, MA, ARIMA, SARIMA, and SARIMAX.
- **5 practical questions** implementing time series decomposition (trend/seasonality/residual) on the AirPassengers dataset, Isolation Forest and Local Outlier Factor for anomaly detection with visualizations, SARIMA forecasting on AirPassengers, and an end-to-end illustrative real-time anomaly detection + short-term forecasting pipeline for a power grid monitoring scenario.
